# Project 1: Thermal Sensation — Day 3: Build a baseline

A **baseline** is a simple model that gives us a score to beat. We will compare a model that always predicts the training-set average with Linear Regression.

We will use group cross-validation on the training buildings. The test buildings remain untouched until the final evaluation notebook.

## 1. Load Project 1 data and the saved building split

Run the Day 1 and Day 2 notebooks first. The split file ensures that this notebook uses the same buildings assigned to training and testing.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor

project_folder = Path.cwd()
if not (project_folder / "data" / "processed").exists():
    project_folder = project_folder.parent

processed_folder = project_folder / "data" / "processed"
cleaned_data_path = processed_folder / "thermal_sensation_cleaned.csv"
split_path = processed_folder / "p1_building_split.csv"

cleaned_data = pd.read_csv(
    cleaned_data_path,
    dtype={"building_id": "string", "subject_id": "string"},
    low_memory=False,
)
split_assignments = pd.read_csv(split_path, dtype={"building_id": "string"})

print("Cleaned data loaded:", len(cleaned_data), "rows")
print("Building split loaded:", len(split_assignments), "buildings")

## 2. Prepare training rows and choose input columns

The target is the numeric `thermal_sensation` vote. Our first feature set uses indoor conditions plus broad global context. We leave IDs, other occupant votes, and calculated comfort indices out of the predictors.

In [ ]:
numeric_features = ["ta", "rh", "vel", "clo", "met"]
categorical_features = [
    "season",
    "country",
    "climate",
    "building_type",
    "cooling_type",
]
selected_features = numeric_features + categorical_features
target_column = "thermal_sensation"

# Attach each building's saved train/test label to the cleaned observations.
model_data = cleaned_data.merge(
    split_assignments,
    on="building_id",
    how="left",
    validate="many_to_one",
)

if model_data["data_split"].isna().any():
    raise ValueError("Some observations do not have a saved train/test assignment.")

training_data = model_data.loc[model_data["data_split"] == "train"].copy()

X_train = training_data[selected_features]
y_train = training_data[target_column]
training_groups = training_data["building_id"]

print("Training rows:", len(training_data))
print("Training buildings:", training_groups.nunique())

## 3. Build a preprocessing pipeline

Some input measurements are missing and some features are text categories. We will fill missing numbers with the training-fold median, scale numeric features for Linear Regression, fill missing categories with the most common training-fold value, and convert categories into indicator columns.

Putting these steps in a pipeline is important: during cross-validation, each fold learns its own fill values from its training portion only.

In [ ]:
numeric_preprocessing = Pipeline(
    steps=[
        ("fill_missing", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)

categorical_preprocessing = Pipeline(
    steps=[
        ("fill_missing", SimpleImputer(strategy="most_frequent")),
        ("make_indicators", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", numeric_preprocessing, numeric_features),
        ("categories", categorical_preprocessing, categorical_features),
    ]
)

## 4. Compare a simple baseline with Linear Regression

`GroupKFold` keeps all observations from a building together inside each fold. We use mean absolute error (MAE); lower MAE means predictions are closer to the observed sensation votes on average.

In [ ]:
group_cross_validation = GroupKFold(n_splits=5)

baseline_model = DummyRegressor(strategy="mean")
linear_regression_model = Pipeline(
    steps=[
        ("prepare_features", preprocessor),
        ("regression", LinearRegression()),
    ]
)

models = {
    "Mean baseline": baseline_model,
    "Linear Regression": linear_regression_model,
}

cross_validation_results = []

for model_name, model in models.items():
    # Scikit-learn returns negative MAE so that larger scores are always better.
    negative_mae_scores = cross_val_score(
        model,
        X_train,
        y_train,
        groups=training_groups,
        cv=group_cross_validation,
        scoring="neg_mean_absolute_error",
    )

    # Change the sign back so the results are easier to understand.
    mae_scores = -negative_mae_scores
    cross_validation_results.append(
        {
            "model": model_name,
            "average_MAE": mae_scores.mean(),
            "standard_deviation": mae_scores.std(),
        }
    )

baseline_results = pd.DataFrame(cross_validation_results)
display(baseline_results.sort_values("average_MAE"))

## Day 3 takeaway

The mean baseline is our first reference score. Linear Regression tests whether a simple relationship between the chosen features and sensation is useful. On the next day, we will compare it with a tree-based model.

These are cross-validation scores from training buildings. We have not used the held-out test buildings.